In [8]:
import sys
import os
import psycopg2
from dotenv import load_dotenv

sys.path.append(os.path.abspath(".."))

load_dotenv()
url = os.getenv("DATABASE_URL")

In [9]:
conn = psycopg2.connect(url, sslmode="require")
cur = conn.cursor()

In [10]:
query = """
SELECT 
    ticker,
    CASE 
        WHEN EXTRACT(DOW FROM MAX(timestamp)) = 3 THEN MAX(timestamp)
        ELSE NULL
    END AS expiry_date
FROM orderbooks
GROUP BY ticker
"""
cur.execute(query)
rows = cur.fetchall()

data = {ticker: expiry for ticker, expiry in rows}
tickers = [ticker for ticker, date in data.items()]

In [11]:
from datetime import datetime, timezone

#sorted by expiry dates
data = dict(sorted(data.items(), key=lambda item: item[1] if item[1] is not None else datetime(2099,1,1, tzinfo=timezone.utc)))

In [12]:
expiry_dates = [date.replace(hour=0, minute=0, second=0, microsecond=0) for ticker, date in data.items() if date is not None]
unique_dates = list(dict.fromkeys(expiry_dates))
print(unique_dates)

[datetime.datetime(2026, 3, 25, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 1, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 8, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 15, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 4, 22, 0, 0, tzinfo=datetime.timezone.utc)]


In [16]:
returns = {}
for date in unique_dates:
    
    #not expiring this day 
    not_expiring_tickers = [ticker for ticker, expiry_date in data.items() 
                             if (expiry_date != None and expiry_date.replace(hour=0, minute=0, second=0, microsecond=0) > date)  
                             or expiry_date == None]

    query = """
        (
            SELECT DISTINCT ON (ticker) *
            FROM orderbooks
            WHERE timestamp >= %s AND timestamp < %s + INTERVAL '1 day'
            AND ticker = ANY(%s)
            ORDER BY ticker, timestamp ASC
        )
        UNION ALL
        (
            SELECT DISTINCT ON (ticker) *
            FROM orderbooks
            WHERE timestamp >= %s AND timestamp < %s + INTERVAL '1 day'
            AND ticker = ANY(%s)
            ORDER BY ticker, timestamp DESC
        );
    """

    cur.execute(query, (date, date, not_expiring_tickers, date, date, not_expiring_tickers))
    rows = cur.fetchall()
    print(rows)
    i = 0
    

[(51203, 'SR270CD6', 'OPTSPOT', datetime.datetime(2026, 3, 25, 6, 0, 45, tzinfo=datetime.timezone.utc), [{'price': 50.33, 'quantity': 800}], [{'price': 50.5, 'quantity': 664}, {'price': 50.67, 'quantity': 800}], 800, 1464), (53231, 'SR270CD6A', 'OPTSPOT', datetime.datetime(2026, 3, 25, 7, 1, 28, tzinfo=datetime.timezone.utc), [{'price': 48.22, 'quantity': 800}], [{'price': 49.61, 'quantity': 800}], 800, 800), (51192, 'SR270CP6', 'OPTSPOT', datetime.datetime(2026, 3, 25, 6, 0, 15, tzinfo=datetime.timezone.utc), [], [{'price': 2.99, 'quantity': 1}], 0, 1), (51191, 'SR270CP6A', 'OPTSPOT', datetime.datetime(2026, 3, 25, 6, 0, 15, tzinfo=datetime.timezone.utc), [], [{'price': 200.49, 'quantity': 1}], 0, 1), (51204, 'SR280CD6', 'OPTSPOT', datetime.datetime(2026, 3, 25, 6, 0, 45, tzinfo=datetime.timezone.utc), [{'price': 40.66, 'quantity': 800}, {'price': 40.51, 'quantity': 800}], [{'price': 40.83, 'quantity': 800}], 1600, 800), (53225, 'SR280CD6A', 'OPTSPOT', datetime.datetime(2026, 3, 25, 7